# 02 · Data Models — One App, Three Shapes

**Part 1 · Data-Intensive Applications** · *Prerequisites: 00–01* · *Book mapping: DDIA ch. 2*

The data model is the deepest decision in a system: it shapes the code you write, the queries
you can ask cheaply, and the mistakes you can make silently. Today we build **the same
application three times** — on a relational store, a document store, and a graph store — and
watch each model make some things trivial and others miserable.

**Learning objectives.**

1. Define the three mainstream models and the *relationship shapes* (one-to-many, many-to-one,
   many-to-many) that decide between them.
2. Demonstrate normalization and denormalization — and produce a real **update anomaly**.
3. Explain schema-on-write vs schema-on-read and where each helps.
4. Contrast declarative and imperative querying on the same task.
5. Choose a model for a given workload, and defend the choice.

Some vocabulary used throughout:

- **Data model**: the abstract shape data is presented in (tables/documents/graphs) — distinct
  from the **storage engine** underneath (nb 03).
- **Impedance mismatch**: the friction between in-memory objects (nested, pointer-rich) and
  the model they must be stored in (flat rows, or trees, or edges).
- **Normalization**: storing each fact once and referencing it by ID. **Denormalization**:
  copying facts near their use for cheap reads — at the price of keeping copies consistent.

In [ ]:
import json
import sqlite3
from collections import Counter, deque

import numpy as np
import pandas as pd

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=2)

## 1. The application and its data

A miniature social app: **users** post short messages, **follow** each other, and read
profiles. The relationships cover every shape that matters:

- user → posts: **one-to-many** (a tree: fits inside a document)
- post → author: **many-to-one** (a reference: wants an ID to point at)
- user ↔ user follows: **many-to-many** (a web: the relational/graph home turf)

We synthesize a community-structured population (people mostly follow within their community,
plus a few celebrities everyone follows) so that graph queries later have real structure to
find.

In [ ]:
N_USERS, COMMUNITY_SIZE, N_CELEBS = 2_000, 40, 5

users = [
    {"user_id": i,
     "handle": f"user{i:04d}",
     "display_name": f"User {i}",
     "community": i // COMMUNITY_SIZE}
    for i in range(N_USERS)
]

# Follow edges: ~15 per user, 80% inside the community, plus the celebrities.
follows = set()
for u in range(N_USERS):
    base = (u // COMMUNITY_SIZE) * COMMUNITY_SIZE
    for _ in range(int(rng.integers(10, 20))):
        if rng.random() < 0.8:  # local edge within the 40-person community
            v = int(base + rng.integers(0, COMMUNITY_SIZE))
        else:                   # long-range edge anywhere
            v = int(rng.integers(0, N_USERS))
        if v != u:
            follows.add((u, v))
    for c in range(N_CELEBS):   # everyone follows the celebrities
        if u != c:
            follows.add((u, c))
follows = sorted(follows)

# Posts: 1-9 per user, with an integer timestamp.
posts, post_id = [], 0
for u in range(N_USERS):
    for j in range(int(rng.integers(1, 10))):
        posts.append({"post_id": post_id, "author_id": u,
                      "ts": int(rng.integers(0, 1_000_000)),
                      "text": f"post {j} from user {u}"})
        post_id += 1

print(f"{len(users):,} users, {len(follows):,} follow edges, {len(posts):,} posts")
assert len(follows) > 20_000 and len(posts) > 7_000

## 2. Relational: everything is a table, relationships are joins

> **Relational model**: data lives in flat **relations** (tables) of unordered **tuples**
> (rows). Relationships are not stored as pointers; they are *reconstructed at query time* by
> **joins** on values (IDs). The schema is enforced on write.

The relational design is fully **normalized**: each fact exists exactly once — a display name
lives in one row, no matter how many posts or followers reference that user.

In [ ]:
db = sqlite3.connect(":memory:")
db.executescript(
    """
    CREATE TABLE users (
        user_id      INTEGER PRIMARY KEY,
        handle       TEXT NOT NULL UNIQUE,
        display_name TEXT NOT NULL,
        community    INTEGER NOT NULL
    );
    CREATE TABLE posts (
        post_id   INTEGER PRIMARY KEY,
        author_id INTEGER NOT NULL REFERENCES users(user_id),
        ts        INTEGER NOT NULL,
        text      TEXT NOT NULL
    );
    CREATE TABLE follows (              -- the many-to-many web, as a plain table
        follower_id INTEGER NOT NULL REFERENCES users(user_id),
        followee_id INTEGER NOT NULL REFERENCES users(user_id),
        PRIMARY KEY (follower_id, followee_id)
    );
    CREATE INDEX idx_posts_author ON posts(author_id, ts);
    CREATE INDEX idx_follows_followee ON follows(followee_id);
    """
)
db.executemany("INSERT INTO users VALUES (?,?,?,?)",
               [(u["user_id"], u["handle"], u["display_name"], u["community"]) for u in users])
db.executemany("INSERT INTO posts VALUES (?,?,?,?)",
               [(p["post_id"], p["author_id"], p["ts"], p["text"]) for p in posts])
db.executemany("INSERT INTO follows VALUES (?,?)", follows)
db.commit()
print("relational store loaded")

Joins make *any* relationship traversable, including ones nobody anticipated at design time.
A timeline (posts by people user 42 follows, newest first) is one query:

In [ ]:
timeline_sql = pd.read_sql_query(
    """
    SELECT p.ts, u.handle, p.text
    FROM follows f
    JOIN posts p ON p.author_id = f.followee_id   -- follow edge -> their posts
    JOIN users u ON u.user_id  = p.author_id      -- attach author info
    WHERE f.follower_id = 42
    ORDER BY p.ts DESC
    LIMIT 5
    """, db)
timeline_sql

And normalization means an update touches exactly one row — every future read, from any query
path, sees the new value:

In [ ]:
db.execute("UPDATE users SET display_name = 'Dr. User 7' WHERE user_id = 7")
(name_via_posts,) = db.execute(
    "SELECT u.display_name FROM posts p JOIN users u ON u.user_id=p.author_id "
    "WHERE p.author_id=7 LIMIT 1").fetchone()
assert name_via_posts == "Dr. User 7", "one UPDATE is visible through every join path"
print("renamed once, consistent everywhere")

## 3. Document: the aggregate is the unit

> **Document model**: each record is a self-contained tree (usually JSON) — schema is *not*
> enforced by the store (**schema-on-read**). One-to-many data nests naturally inside its
> parent, giving **locality**: one read fetches the whole aggregate.

Our document store is honest-to-its-model: a dict of `user_id → JSON document`, each embedding
the user's posts (one-to-many: perfect) and their follow list (many-to-many: watch closely).
To make profile pages render without joins, we also *denormalize* each followee's display name
into the follower's document — a very common real-world choice.

In [ ]:
display_name = {u["user_id"]: u["display_name"] for u in users}
display_name[7] = "Dr. User 7"  # keep parity with the relational store

doc_store: dict[int, str] = {}
posts_by_author: dict[int, list] = {}
for p in posts:
    posts_by_author.setdefault(p["author_id"], []).append(
        {"ts": p["ts"], "text": p["text"]})
followees: dict[int, list[int]] = {}
for a, b in follows:
    followees.setdefault(a, []).append(b)

for u in users:
    uid = u["user_id"]
    doc_store[uid] = json.dumps({
        "handle": u["handle"],
        "display_name": display_name[uid],
        "community": u["community"],
        "posts": sorted(posts_by_author.get(uid, []), key=lambda p: -p["ts"]),
        "following": [
            # DENORMALIZED: we copy the display name so the profile page needs
            # no second lookup. Remember this decision.
            {"user_id": v, "display_name": display_name[v]}
            for v in followees.get(uid, [])
        ],
    })
print(f"document store loaded: {len(doc_store):,} documents")

**The locality win.** A profile page (user + their posts) is a single key-value fetch — no
joins, no second query, and the document arrives in exactly the shape the application renders
(no impedance mismatch). Compare the work:

In [ ]:
def profile_document(uid: int) -> tuple[dict, int]:
    """Fetch a profile page. Returns (profile, store round trips used)."""
    return json.loads(doc_store[uid]), 1       # one lookup, whole aggregate

def profile_relational(uid: int) -> tuple[dict, int]:
    (handle, name) = db.execute(
        "SELECT handle, display_name FROM users WHERE user_id=?", (uid,)).fetchone()
    rows = db.execute(
        "SELECT ts, text FROM posts WHERE author_id=? ORDER BY ts DESC", (uid,)).fetchall()
    return {"handle": handle, "display_name": name,
            "posts": [{"ts": t, "text": x} for t, x in rows]}, 2

# Same answer from both stores...
(d, doc_trips), (r, rel_trips) = profile_document(1234), profile_relational(1234)
assert d["handle"] == r["handle"] and d["posts"] == r["posts"]

# ...but they differ in ROUND TRIPS to the store. In production, every round
# trip is a datacenter network hop (~250 µs on nb 01's latency ladder), and
# round trips - not CPU - dominate simple reads. Model that cost explicitly:
RTT_MS = 0.25
print(f"profile page - document: {doc_trips} round trip ({doc_trips*RTT_MS:.2f} ms network), "
      f"relational: {rel_trips} round trips ({rel_trips*RTT_MS:.2f} ms network)")
assert doc_trips < rel_trips, "aggregate locality: the whole page in one fetch"

**Failure first: the update anomaly.** Now user 3 — a celebrity, followed by everyone —
changes their display name. In the relational store that would be one row. Here, the name is
*copied into every follower's document*. An update that forgets the copies — the classic bug —
leaves the database disagreeing with itself:

In [ ]:
# The naive update: fix the celebrity's own document only.
own = json.loads(doc_store[3])
own["display_name"] = "The Artist"
doc_store[3] = json.dumps(own)

# Any follower's page still shows the old name:
some_follower = next(a for a, b in follows if b == 3 and a != 3)
stale = [f for f in json.loads(doc_store[some_follower])["following"]
         if f["user_id"] == 3][0]
print(f"user {some_follower}'s page shows: {stale['display_name']!r}  (truth: 'The Artist')")
assert stale["display_name"] != "The Artist", "denormalized copies went stale - anomaly!"

# The real fix costs a write to EVERY follower document:
touched = 0
for uid, raw in doc_store.items():
    doc = json.loads(raw)
    changed = False
    for f in doc["following"]:
        if f["user_id"] == 3:
            f["display_name"] = "The Artist"
            changed = True
    if changed:
        doc_store[uid] = json.dumps(doc)
        touched += 1
print(f"repair required rewriting {touched:,} documents")
assert touched > 1000, "one logical change fanned out into thousands of writes"

That is the denormalization contract in one cell: **reads got cheaper because writes took on
the job of keeping every copy correct.** It can absolutely be the right trade (read-heavy
systems make it constantly — see nb 12 and capstone 23) — but it must be *chosen*, with the
repair path designed, not stumbled into. The safer default inside documents: store **IDs**
(stable, meaning-free identifiers) and join in the application when you must:

In [ ]:
def timeline_document(uid: int, limit: int = 5) -> list[dict]:
    """A timeline on the document store = an application-side join:
    fetch my document, then fetch every followee's document. N+1 lookups."""
    me = json.loads(doc_store[uid])
    items = []
    for f in me["following"]:
        who = json.loads(doc_store[f["user_id"]])
        items += [{"ts": p["ts"], "handle": who["handle"], "text": p["text"]}
                  for p in who["posts"]]
    return sorted(items, key=lambda p: -p["ts"])[:limit]

assert (pd.DataFrame(timeline_document(42)).values == timeline_sql.values).all(), \
    "app-side join reproduces the SQL join - with N+1 fetches of code we now maintain"
print("document timeline matches SQL timeline (at N+1 lookup cost)")

### Schema-on-read, demonstrated in one cell

Add a `verified` flag to *new* documents only. The document store doesn't blink — old and new
documents coexist, and the **reader** supplies the interpretation (a default). The relational
store instead migrates once (`ALTER TABLE`) and the schema stays uniform. Schema-on-read is
flexibility; schema-on-write is a guarantee. (Notebook 04 turns this into a full theory of
schema evolution.)

In [ ]:
doc_store[9999_1] = json.dumps({"handle": "newcomer", "display_name": "Newcomer",
                                "community": -1, "posts": [], "following": [],
                                "verified": True})

def is_verified(uid: int) -> bool:
    return json.loads(doc_store[uid]).get("verified", False)  # reader-side default

assert is_verified(9999_1) is True and is_verified(42) is False
db.execute("ALTER TABLE users ADD COLUMN verified INTEGER NOT NULL DEFAULT 0")
assert db.execute("SELECT verified FROM users WHERE user_id=42").fetchone() == (0,)
print("schema-on-read (per-document) vs schema-on-write (one migration): both handled")

## 4. Graph: relationships are the data

> **Property graph model**: **vertices** (with properties) connected by directed, labeled
> **edges** (with properties). Anything can connect to anything; queries *walk* the structure,
> following edges a variable number of hops — the case both other models handle worst.

In [ ]:
class GraphStore:
    """A minimal property graph: vertices, labeled edges, and traversals."""

    def __init__(self):
        self.vertex: dict[int, dict] = {}
        self.out_edges: dict[tuple[int, str], set[int]] = {}
        self.in_edges: dict[tuple[int, str], set[int]] = {}

    def add_vertex(self, vid: int, **props):
        self.vertex[vid] = props

    def add_edge(self, src: int, label: str, dst: int):
        if src not in self.vertex or dst not in self.vertex:
            raise KeyError(f"edge {src}-[{label}]->{dst} references a missing vertex")
        self.out_edges.setdefault((src, label), set()).add(dst)
        self.in_edges.setdefault((dst, label), set()).add(src)

    def out(self, vid: int, label: str) -> set[int]:
        return self.out_edges.get((vid, label), set())

    def shortest_path(self, src: int, dst: int, label: str) -> list[int] | None:
        """Breadth-first search along `label` edges - variable-length traversal,
        the graph model's home turf."""
        if src == dst:
            return [src]
        parent, frontier, seen = {}, deque([src]), {src}
        while frontier:
            v = frontier.popleft()
            for w in self.out(v, label):
                if w in seen:
                    continue
                parent[w] = v
                if w == dst:  # walk parents back to recover the path
                    path = [dst]
                    while path[-1] != src:
                        path.append(parent[path[-1]])
                    return path[::-1]
                seen.add(w)
                frontier.append(w)
        return None

    def suggest_follows(self, vid: int, k: int = 5) -> list[tuple[int, int]]:
        """'People you may know': friends-of-friends I don't yet follow,
        ranked by how many of my followees follow them (mutual count)."""
        mine = self.out(vid, "follows")
        counts = Counter(
            w
            for v in mine
            for w in self.out(v, "follows")
            if w != vid and w not in mine
        )
        return counts.most_common(k)

g = GraphStore()
for u in users:
    g.add_vertex(u["user_id"], handle=u["handle"])
for a, b in follows:
    g.add_edge(a, "follows", b)

path = g.shortest_path(1990, 250, "follows")
print(f"how user 1990 reaches user 250 through follows: {path}")
assert path is not None and path[0] == 1990 and path[-1] == 250
assert all(path[i + 1] in g.out(path[i], "follows") for i in range(len(path) - 1)), \
    "every hop of the path is a real edge"

suggestions = g.suggest_follows(42)
print(f"suggested follows for user 42 (id, mutuals): {suggestions}")
assert all(s not in g.out(42, "follows") for s, _ in suggestions), \
    "never suggest someone already followed"
assert suggestions[0][1] >= suggestions[-1][1], "ranked by mutual count"

Could SQL do this? Yes — relational is *capable* of graphs (edges are just a table), and SQL
even has recursive queries. But compare the shapes: BFS above reads like the algorithm;
the SQL version below is a fixed-depth join pyramid that must be rewritten for every depth,
and a recursive CTE for the general case is famously easy to get subtly wrong. **When the
query is a walk, use a model that walks.**

In [ ]:
# Depth-2 reachability, SQL edition (one JOIN per hop - depth is hardcoded):
fof_sql = {row[0] for row in db.execute(
    """
    SELECT DISTINCT f2.followee_id
    FROM follows f1 JOIN follows f2 ON f2.follower_id = f1.followee_id
    WHERE f1.follower_id = 42
    """)}
fof_graph = {w for v in g.out(42, "follows") for w in g.out(v, "follows")}
assert fof_sql == fof_graph, "same answer, very different ergonomics"
print(f"friends-of-friends of user 42: {len(fof_graph)} users (SQL == graph traversal)")

## 5. Declarative vs imperative querying

SQL is **declarative**: you state *what* you want (the shape of the result) and the system
picks *how* (index vs scan, join order, parallelism). Imperative code fixes the *how* forever.
Same task both ways — top five posters:

In [ ]:
top_sql = db.execute(
    """
    SELECT author_id, COUNT(*) AS n FROM posts
    GROUP BY author_id ORDER BY n DESC, author_id LIMIT 5
    """).fetchall()

counts = Counter(p["author_id"] for p in posts)                  # the 'how', spelled out
top_imperative = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:5]

assert top_sql == top_imperative
print("declarative == imperative result:", top_sql)

The declarative form is shorter — but the deep advantage is **freedom for the system**: the
optimizer can change strategy when data grows or an index appears, *without touching your
code*. This idea scales far beyond SQL: it is why dataflow engines (nb 10), streaming SQL
(nb 11), and even ML feature definitions (nb 16) all chase declarative interfaces.

## 6. Choosing a model

| Workload trait | Relational | Document | Graph |
|---|---|---|---|
| Self-contained one-to-many aggregates (order + items, profile + posts) | fine | **best: locality, no mismatch** | fine |
| Many-to-many everywhere; ad-hoc questions not known in advance | **best: joins + optimizer** | painful (app-side joins) | good |
| Variable-depth relationships (paths, influence, recommendations) | painful (recursive SQL) | worst | **best: traversal is native** |
| Schema needs to vary per record / evolve without migrations | migrations | **schema-on-read** | flexible |
| Strong write-time integrity (foreign keys, constraints) | **best** | weakest | varies |

Three honest caveats to the table. First, **models converge**: relational stores speak JSON
(schema-on-read inside a column), document stores grew joins and schema validation; you often
pick a *primary* shape, not a prison. Second, the storage engine underneath (nb 03) and the
replication design above (nb 05) often matter more to performance than the model. Third,
denormalization is not a model property but a *decision* — you can make it (and its update
anomaly) in any model; you saw the whole trade in §3.

## 7. Exercises

1. **Model it three ways.** A university has students, courses, and enrollments (a student
   takes many courses; a course has many students; an enrollment has a grade). Sketch the
   relational schema, a document design (whose document owns the enrollment?), and a graph
   design. Which single query is most awkward in each?
2. **Find the anomaly.** A delivery app stores each order as a document embedding the
   restaurant's name and address. The restaurant moves. What breaks, what is the repair, and
   when is this denormalization nevertheless correct? (Hint: think about what an old *order*
   should show.)
3. **Timeline depth.** Extend `timeline_document` to depth 2 (posts from followees *and their*
   followees). How many document fetches does it take for user 42? What does the same change
   cost in SQL?
4. **Recursive SQL.** Using `WITH RECURSIVE`, compute every user reachable from user 1990
   within 3 follow-hops, and verify it against a BFS on the graph store.
5. **Design call.** Your product is "LinkedIn for labs": profiles (rarely updated, heavily
   read), a dense collaboration network (path queries drive the product), and full-text
   publications. Argue a polyglot design: which data goes in which model, and what is the
   source of truth?

### Solutions

**1.** Relational: `students`, `courses`, `enrollments(student_id, course_id, grade)` — all
queries easy. Document: the enrollment must live *somewhere* — inside the student (course
rosters become scatter-gathers) or inside the course (transcripts do); either way one
direction of the many-to-many suffers. Graph: `(:Student)-[:ENROLLED {grade}]->(:Course)` —
both directions easy, but "GPA per student per semester" becomes a traversal + aggregation
that SQL would do in one GROUP BY.

**2.** Old orders now show a stale address — and that is (partly) **correct**: an order is a
*historical fact*; it should show the address *as it was at order time*. The anomaly and the
feature are the same mechanism. The repair, if you do want current data: store the restaurant
ID and join at read time for live fields, keep embedded copies only for point-in-time fields.
Distinguishing "facts about now" from "facts about then" is a recurring design skill (it
returns as event sourcing in nb 11 and point-in-time features in capstone 25).

**3.** Depth 2 for user 42 ≈ 1 + |following| + Σ|their following| fetches — hundreds to
thousands (measured below). SQL: still one query — add one more join. Fan-out moved from the
database to a hand-written N+1 loop; this is why "we'll join in the app" quietly becomes a
performance program.

**4 & 5.** Code below for 4. For 5: profiles → document store (aggregate reads, flexible
fields); collaboration network → graph (path queries are the product); publications → search
index (nb 12 builds one); **source of truth**: one primary store (or an event log, nb 11)
from which index and caches are *derived* — never let three stores each believe they are the
truth.

In [ ]:
# Exercise 3 (measure the fan-out):
me = json.loads(doc_store[42])
depth2_fetches = 1 + len(me["following"]) + sum(
    len(json.loads(doc_store[f["user_id"]])["following"]) for f in me["following"])
print(f"depth-2 document timeline for user 42: {depth2_fetches:,} fetches")
assert depth2_fetches > 200

# Exercise 4: recursive CTE vs BFS - same reachable set within 3 hops.
reachable_sql = {row[0] for row in db.execute(
    """
    WITH RECURSIVE reach(user_id, depth) AS (
        SELECT ?, 0
        UNION
        SELECT f.followee_id, r.depth + 1
        FROM reach r JOIN follows f ON f.follower_id = r.user_id
        WHERE r.depth < 3
    )
    SELECT DISTINCT user_id FROM reach
    """, (1990,))}

frontier, seen = {1990}, {1990}
for _ in range(3):
    frontier = {w for v in frontier for w in g.out(v, "follows")} - seen
    seen |= frontier
assert reachable_sql == seen, "recursive SQL == BFS"
print(f"users reachable from 1990 within 3 hops: {len(seen):,} (SQL == graph)")

## 8. Takeaways

- Choose by **relationship shape**: trees → documents (locality, no impedance mismatch);
  webs of many-to-many + ad-hoc questions → relational (joins + optimizer); variable-depth
  walks → graph (traversal is native).
- **Normalization** is an integrity technology: one fact, one place. **Denormalization** is a
  read-optimization that transfers the consistency burden to every future write — you watched
  the anomaly happen; only make the trade with the repair path designed.
- **Schema-on-write** buys guarantees; **schema-on-read** buys evolution. You will need the
  full evolution toolkit either way (nb 04).
- Declarative interfaces hand the *how* to the system — the single most reused idea in data
  infrastructure.
- Models converge; pick a primary shape, and let other shapes be *derived* views (nb 12).

**Further reading.** DDIA ch. 2; Codd, *A Relational Model of Data for Large Shared Data
Banks* (1970); the Neo4j Cypher docs for real graph querying.

**Next:** [03 · Storage engines](03-storage-engines.ipynb) — beneath every model: how bytes
actually get to disk and back, fast. We build a key-value engine three ways.